# ViGoEmotions — Kaggle: 28 nhãn, gồm Neutral

So sánh TF-IDF + OvR LR, BiGRU và PhoBERT với baseline 27 nhãn.
Attach hai Dataset Private: ZIP code 28 nhãn và ba CSV train/val/test.
Trong Settings/Session options, bật GPU + Internet; không chọn TPU.
Chạy từng cell từ trên xuống; dừng khi có lỗi. Hướng dẫn: docs/neutral_28_labels.md.
Artifact ở /kaggle/working/ViGoEmotions28/reports/neutral_28_labels/.
Model hoàn tất được giữ; không hỗ trợ resume giữa epoch.


## 1. Nạp project từ Kaggle Input

Attach Dataset chứa **vigoemotions_neutral28_colab.zip** mới, không dùng ZIP 04/05.
Hỗ trợ cả ZIP và Dataset đã được Kaggle giải nén. Code tự tìm đường dẫn.
Cell dưới chỉ chạy một lần trong workspace mới; không giải nén đè kết quả.
Sau restart kernel, bỏ qua cell giải nén nếu project còn và chạy cell nối workspace.


In [ ]:
from pathlib import Path
import os, shutil, zipfile

INPUT = Path('/kaggle/input')
ROOT = Path('/kaggle/working/ViGoEmotions28')
if ROOT.exists():
    raise FileExistsError(f'{ROOT} đã tồn tại. Không giải nén đè lên kết quả; chạy tiếp cell cần thiết hoặc dùng session mới.')

# Kaggle có thể giữ ZIP hoặc tự giải nén ZIP khi tạo Dataset.
archives = list(INPUT.rglob('vigoemotions_neutral28_colab.zip'))
code_roots = [p.parent.parent for p in INPUT.rglob('scripts/neutral_28_labels.py')]
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as archive:
        for item in archive.infolist():
            if not (ROOT / item.filename).resolve().is_relative_to(ROOT.resolve()):
                raise ValueError('Unsafe ZIP path')
        ROOT.mkdir(parents=True)
        archive.extractall(ROOT)
elif not archives and len(code_roots) == 1:
    shutil.copytree(code_roots[0], ROOT)
else:
    raise RuntimeError('Hãy attach đúng một Dataset code chứa vigoemotions_neutral28_colab.zip hoặc project đã giải nén.')
os.chdir(ROOT)
print('Project:', ROOT)


In [ ]:
# Cũng dùng cell này để nối lại workspace sau khi restart kernel.
from pathlib import Path
import os, sys, subprocess
ROOT = Path('/kaggle/working/ViGoEmotions28')
assert (ROOT/'scripts/neutral_28_labels.py').is_file(), 'Thiếu module 28 nhãn; kiểm tra ZIP/version repo.'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'scripts'))


## 2. Dependency tối thiểu và GPU
Không cài bộ lock local của macOS. Không cần reportlab/pypdf vì báo cáo thí nghiệm là Markdown.
Nếu pip yêu cầu restart kernel, restart rồi chạy lại cell nối workspace, GPU, helper và các bước sau;
không cần upload/giải nén lại. Các lệnh cài đặt có kiểm tra exit code.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=1.26,<3', 'pandas>=2.2,<4', 'scipy>=1.13,<2', 'scikit-learn>=1.6,<2',
    'matplotlib>=3.9,<4', 'torch>=2.6,<3', 'transformers==4.57.3',
    'py-vncorenlp==0.1.4', 'joblib', 'threadpoolctl', 'tqdm'], check=True)
import shutil
if shutil.which('java') is None:
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'default-jre-headless'], check=True)


In [ ]:
import torch, subprocess
print('Python:', sys.version)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('TF-IDF vẫn chạy được; bật GPU trước BiGRU/PhoBERT.')
subprocess.run(['java', '-version'], check=True)


## 3. Dữ liệu và quy tắc annotation
Attach Dataset Private chứa `train.csv`, `val.csv`, `test.csv` có quyền truy cập. Không đưa dữ liệu nguồn vào repo công khai.
Giữ split gốc; áp dụng cùng quy tắc loại train trùng heldout của pipeline 27 nhãn (không tạo split mới).
Test text chỉ được dùng để audit overlap như code gốc; không dùng test target để fit/chọn epoch/ngưỡng.

**Target 28 chiều:** `[27]` → chỉ Neutral = 1; `[2,27]` → Joy = 1 và Neutral = 1.
Giữ đầy đủ nhãn gốc; ID lặp gộp khi multi-hot. Pipeline 27 nhãn cũ không thay đổi.
Guideline do người dùng cung cấp nói Neutral không đi cùng cảm xúc, nhưng dữ liệu có **306 mixed**
và **725 Neutral-only** (tổng trước loại overlap). Chúng ta giữ annotation vì chưa biết nhãn nào đúng.
Không tự sửa hay loại các mixed sample. Nhánh ablation bổ sung không phải điều kiện hoàn thành study này.


In [ ]:
from pathlib import Path
import csv, shutil

sources = {}
for name in ('train.csv', 'val.csv', 'test.csv'):
    candidates = list(Path('/kaggle/input').rglob(name))
    if len(candidates) != 1:
        raise RuntimeError(f'Cần đúng một {name} trong Input; tìm thấy {len(candidates)}. Kiểm tra Dataset đã attach.')
    with candidates[0].open(encoding='utf-8-sig', newline='') as handle:
        if next(csv.reader(handle), None) != ['id', 'text', 'labels']:
            raise ValueError(f'{name}: schema phải là id,text,labels')
    sources[name] = candidates[0]
(ROOT / 'data').mkdir(exist_ok=True)
for name, source in sources.items():
    shutil.copy2(source, ROOT / 'data' / name)
    print('Đã nạp:', name)

DATA = ROOT / 'data'


## 4. Helper và tạo study riêng
Sao chép settings từ `configs/midterm.json` đi kèm ZIP; kiểm tra epoch trong config in ra sau init.
Config hiện tại có thể khác lượt baseline 27 nhãn đã lưu; không tự lấy epoch từ notebook khác.
Nếu baseline bạn muốn đối chiếu dùng epoch khác, hãy cung cấp đúng config/run/report của baseline đó
và tạo **study mới**; không sửa config gốc. Điều kiện khác môi trường, epoch và checksum được ghi trong bảng.
Từng model lưu toàn bộ config thực dùng, environment, thời gian và checkpoint selection.


In [ ]:
import neutral_28_labels as n28
from IPython.display import display
import pandas as pd
import json

OUT = ROOT/'reports/neutral_28_labels'
BASE_CONFIG = ROOT/'configs/midterm.json'
BASELINE_REPORTS = ROOT/'reports'  # Có thể trỏ đến reports 27 nhãn khác, chỉ đọc.

# Có thể đổi OUT thành ROOT/'reports/neutral_28_labels/study_moi' trước khi init.
# Các lựa chọn được đóng băng khi init, không sửa giữa study.
def run28(action, *arguments):
    subprocess.run([sys.executable, '-u', str(ROOT/'scripts/neutral_28_labels.py'), action,
                    '--output', str(OUT), '--data-dir', str(DATA), *map(str, arguments)],
                   cwd=ROOT, check=True)

run28('init', '--base-config', BASE_CONFIG, '--baseline-reports', BASELINE_REPORTS)
display(pd.read_csv(OUT/'annotation_counts.csv'))
print('Config:', json.loads((OUT/'config.json').read_text()))
assert n28.LABELS28[27] == 'neutral' and len(n28.LABELS28) == 28
assert n28.multihot28([27]).tolist() == [0]*27 + [1]
assert n28.multihot28([2,27])[2] == n28.multihot28([2,27])[27] == 1


## 5. Smoke tests offline
Kiểm tra target/metrics, fit TF-IDF và BiGRU nhỏ, backward/nạp lại checkpoint và một RoBERTa tiny
cùng kiểu head với PhoBERT. Mô hình tiny chỉ kiểm tra kỹ thuật, không phải pretrained PhoBERT và không
được ghi vào kết quả thí nghiệm. Test này không phụ thuộc config 4 epoch cố định của bộ test cũ.


In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
                '-p', 'test_neutral_28_labels.py', '-v'], cwd=ROOT, check=True)


## 6. Model A — TF-IDF + One-vs-Rest Logistic Regression
Giữ nguyên TF-IDF, normalization và LogisticRegression của config gốc; fit vocabulary/IDF chỉ trên train.
28 classifier nhị phân độc lập, bao gồm Neutral. Đánh giá validation tại 0.5; lưu cả vectorizer và model
để inference không fit lại. Artifact riêng ở `models/tfidf_lr/` bên trong OUT.


In [ ]:
run28('train', '--model', 'tfidf_lr')
display(pd.read_csv(OUT/'metrics_summary.csv'))


## 7. Model B — BiGRU
Tái dùng BiGRU/forward gốc; chỉ thay head 27 → 28. Vocabulary từ train, lowercase/whitespace,
padding/truncation như cũ. BCEWithLogitsLoss, AdamW, gradient clipping 1.0, fixed seed và shuffled DataLoader
như baseline. Chọn checkpoint bằng validation **28-label Macro-F1 tại 0.5**, hòa chọn epoch sớm nhất.
Hiện tiến độ từng batch và metric mỗi epoch. Không dùng test để chọn epoch.


In [ ]:
assert torch.cuda.is_available(), 'Bật GPU runtime trước khi chạy.'
run28('train', '--model', 'gru')
display(pd.read_csv(OUT/'models/gru/history.csv'))


## 8. Model C — PhoBERT
Giữ pretrained checkpoint/revision của baseline, VnCoreNLP wseg trước BPE và tokenizer use_fast=False.
Head 28 logits; optimizer nhận toàn bộ backbone + head (không freeze). BCEWithLogitsLoss và ngưỡng 0.5.
Tải pretrained/segmenter lần đầu cần Internet/Java. Checkpoint và tokenizer được lưu riêng để reload.
Cảnh báo head mới khởi tạo là bình thường. Mọi metric in ra là số đo thật.


In [ ]:
assert torch.cuda.is_available(), 'Bật GPU runtime trước khi chạy.'
run28('train', '--model', 'phobert')
display(pd.read_csv(OUT/'models/phobert/history.csv'))


## 9. Kết quả và so sánh công bằng
Bảng 6 dòng lấy điểm 27 nhãn thật từ reports; nếu thiếu sẽ ghi missing, không dựng số liệu.
Các cột metric chính của 28 nhãn chấm cả Neutral; **emotion27_*** chấm cùng 27 cảm xúc để đối chiếu.
Không diễn giải việc đổi tập nhãn là cải thiện. Điều kiện khác config/runtime/data và xác minh ID/target
được ghi trong `conditions`; nguồn 27 nhãn chỉ đọc, không ghi đè. LRAP của 27 nhãn cho neutral-only
(all-zero target) điểm 1 là một khác biệt cần nêu khi so với LRAP 28 nhãn.


In [ ]:
run28('report')
for name in ('metrics_summary.csv', 'neutral_analysis.csv', 'comparison_27_vs_28.csv',
             'emotion27_per_class_comparison.csv', 'per_class_metrics.csv'):
    print(name)
    display(pd.read_csv(OUT/name))
from IPython.display import Markdown
display(Markdown((OUT/'experiment_report.md').read_text(encoding='utf-8')))


## 10. Inference trên comment mới, từ checkpoint đã lưu
Bạn có thể thay danh sách COMMENTS. Đây là câu ví dụ tự viết, không khẳng định trước model sẽ dự đoán loại nào.
Mỗi model hiển thị văn bản gốc, cảm xúc dự đoán, Neutral, xác suất cho mọi nhãn vượt 0.5 và top 5 kể cả
không nhãn nào vượt ngưỡng. Không ép Neutral loại trừ cảm xúc: primary study cho phép đồng dự đoán.
Checkpoint pickle/joblib chỉ nạp từ nguồn của chính bạn. File ví dụ chứa text nhập vào nên cũng giữ private.


In [ ]:
COMMENTS = [
    'Cuộc họp bắt đầu lúc tám giờ sáng thứ hai.',
    'Mình vui quá, cuối cùng cũng làm được rồi!',
    'Vừa vui vì gặp lại bạn, vừa buồn vì sắp phải chia tay.',
    'Mình hiểu rồi, nhưng vẫn hơi thất vọng về kết quả.',
]
summary, details = n28.infer_examples(OUT, COMMENTS)
display(summary)
for (model, index), table in details.groupby(['model', 'input_index'], sort=False):
    print(model, '| Input:', COMMENTS[index])
    display(table[['label', 'probability', 'predicted', 'top5']])


### Tìm ví dụ thật cho từng kiểu dự đoán
Tìm trên xác suất validation đã lưu: Neutral-only, một cảm xúc, nhiều cảm xúc, Neutral + cảm xúc.
Nếu một kiểu không xuất hiện, báo `available_count=0`; không hardcode hay điều chỉnh threshold để tạo ca mong muốn.
DISPLAY_PRIVATE_TEXT mặc định bật để xem input và xác suất của ca tìm được trong runtime riêng tư.
CSV chỉ xuất ID/số lượng, không xuất comment gốc. Xóa output chứa text trước khi chia sẻ notebook.


In [ ]:
DISPLAY_PRIVATE_TEXT = True  # Chỉ hiển thị trong runtime riêng tư; không chia sẻ output có comment nguồn.
cases = n28.find_validation_cases(OUT, DATA, display_text=DISPLAY_PRIVATE_TEXT)
display(cases)


## 11. Test cuối cùng — tùy chọn, sau khi khóa mọi lựa chọn
Mặc định **không chạy test**, tương thích protocol midterm 27 nhãn giữ test cho Final.
Nếu đã chốt model/hyperparameters/threshold, đặt RUN_FINAL_TEST=True. Cả ba model phải hoàn tất.
Bước này khóa checksum checkpoint và cấm train tiếp trong study, chỉ chấm test tại 0.5.
Không chọn mô hình/ngưỡng/epoch theo test. Nếu baseline 27 nhãn chỉ có val thì không tạo so sánh test giả.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    run28('test')
    display(pd.read_csv(OUT/'metrics_summary.csv'))
else:
    print('Test chưa đánh giá. Chỉ dùng validation cho so sánh primary hiện tại.')


## 12. Lưu kết quả và checkpoint riêng tư

Chạy sau inference/report. ZIP toàn bộ study gồm CSV, NPZ, config, checkpoint,
tokenizer, vocabulary và segmenter; không chứa CSV/comment nguồn từ data/.
Tải ZIP từ file browser/Output trong /kaggle/working trước khi kết thúc session.
Giữ ZIP và notebook Private. Save & Run All sẽ chạy lại toàn bộ trong session mới.


In [ ]:
import zipfile
archive_path = Path('/kaggle/working/vigoemotions_neutral28_private_results.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in OUT.rglob('*'):
        if path.is_file():
            archive.write(path, str(Path('neutral_28_labels')/path.relative_to(OUT)))
print('Private results:', archive_path, archive_path.stat().st_size/1024**2, 'MiB')
print('Tải ZIP từ file browser/Output:', archive_path)
